In [45]:
import numpy as np
import cv2
# import imutils
import os
from os.path import join
import math
import matplotlib.pyplot as plt

In [56]:
DATASET = './evs_mot_public_dataset'
TRAIN = os.path.join(DATASET, "evs_mot-train")
SEQUENCE = "MOT_02"

img = os.path.join(TRAIN, SEQUENCE, 'img1/%06d.jpg')
det_file = os.path.join(TRAIN, SEQUENCE, 'det', 'det.txt')
gt_file = os.path.join(TRAIN, SEQUENCE, 'gt', 'gt.txt')

In [ ]:
def calculate_iou(boxA,boxB):
    boxA_x1, boxA_y1 = boxA[0], boxA[1]
    boxA_x2, boxA_y2 = boxA[0] + boxA[2], boxA[1] + boxA[3]
    boxB_x1, boxB_y1 = boxB[0], boxB[1]
    boxB_x2, boxB_y2 = boxB[0] + boxB[2], boxB[1] + boxB[3]

    xA = max(boxA_x1, boxB_x1)
    yA = max(boxA_y1, boxB_y1)
    xB = min(boxA_x2, boxB_x2)
    yB = min(boxA_y2, boxB_y2)

    width_iou = max(0, xB - xA)
    height_iou = max(0, yB - yA)
    interArea = width_iou * height_iou
    boxAreaA = boxA[2] * boxA[3]
    boxAreaB = boxB[2] * boxB[3]
    iou = interArea / (boxAreaA + boxAreaB - interArea)
    return iou

def calculate_distance(boxA, boxB):
    cx_A = boxA[0] + boxA[2] / 2
    cy_A = boxA[1] + boxA[3] / 2

    cx_B = boxB[0] + boxB[2] / 2
    cy_B = boxB[1] + boxB[3] / 2
    
    distance = ((cx_A - cx_B)**2 + (cy_A - cy_B)**2)**0.5
    
    return distance

In [4]:
# raw_det = np.loadtxt(det_file, delimiter=',')
# detections_dict = {}

# for row in raw_det:
#     f = int(row[0])
#     conf = row[6]
#     if conf < 0.4:
#         continue
#     if f not in detections_dict:
#         detections_dict[f] = []
#     detections_dict[f].append(row[2:6])

# active_tracks = {}
# next_id = 1
# lost_nr = 25
# cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
# frame_nr = 1

# while True:
#     ret, frame = cap.read()

#     if not ret:
#         break

#     current_det = detections_dict.get(frame_nr, [])
#     new_tracks = {}
#     used_tracks = set()

#     for det in current_det:
#         best_id = None
#         max_iou = 0.25

#         for t_id, track in active_tracks.items():

#             if t_id in used_tracks:
#                 continue

#             t_bbox = track["bbox"]
#             iou = calculate_iou(det, t_bbox)

#             if iou > max_iou:

#                 max_iou = iou
#                 best_id = t_id

#         if best_id is not None:
#             new_tracks[best_id] = {"bbox": det, "lost" :0}
#             used_tracks.add(best_id)

#         else:
#             new_tracks[next_id] = {"bbox": det, "lost" :0}
#             next_id += 1

#     for t_id, track in active_tracks.items():

#         if t_id not in used_tracks:
#             track["lost"] += 1

#             if track["lost"] <= lost_nr:
#                 new_tracks[t_id] = track

#     active_tracks = new_tracks
#     for t_id, track in active_tracks.items():
#         bbox = track["bbox"]

#         if track["lost"] > 0:
#             continue
#         x, y, w, h = map(int, bbox)
#         cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)
#         cv2.putText(frame, f'ID: {t_id}', (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

#     cv2.imshow('Tracking', frame)
#     frame_nr += 1
#     if cv2.waitKey(15) & 0xFF == ord('q'): break

# cap.release()
# cv2.destroyAllWindows()

In [57]:
raw_det = np.loadtxt(det_file, delimiter=',')

detections_dict = {}
active_tracks = {}

results_list = []

next_id = 1
lost_nr = 100
cap = cv2.VideoCapture(img, cv2.CAP_IMAGES)
frame_nr = 1
min_conf = 0.15
new_track_conf = 0.55

for row in raw_det:
    f = int(row[0])
    conf = row[6]
    if conf < min_conf:
        continue
    if f not in detections_dict:
        detections_dict[f] = []
    detections_dict[f].append({
        "bbox": row[2:6],
        "conf": conf
    })

while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_det = detections_dict.get(frame_nr, [])
    new_tracks = {}
    used_tracks = set()

    for detection in current_det:
        det = detection["bbox"]
        conf = detection["conf"]

        best_id = None
        best_score = -float('inf')

        for t_id, track in active_tracks.items():
            if t_id in used_tracks:
                continue

            t_bbox = track["bbox"]
            iou = calculate_iou(det, t_bbox)
            distance = calculate_distance(det, t_bbox)

            if iou < 0.1:
                continue
            score = iou * 0.5 + conf * 0.25 - (distance / 500) * 0.25

            if score > best_score:
                best_score = score
                best_id = t_id

        if best_id is not None and best_score > 0.25:
            new_tracks[best_id] = {"bbox": det, "lost": 0, "conf":conf}
            used_tracks.add(best_id)
        else:
            if conf >= new_track_conf:
                new_tracks[next_id] = {"bbox": det, "lost": 0, "conf":conf}
                next_id += 1

    for t_id, track in active_tracks.items():
        if t_id not in used_tracks:
            track["lost"] += 1
            if track["lost"] < lost_nr:
                new_tracks[t_id] = track

    active_tracks = new_tracks
    
    for t_id, track in active_tracks.items():
        if track["lost"] == 0:
            bbox = track["bbox"]
            x, y, w, h = bbox
            results_list.append(f"{frame_nr},{t_id},{x:.1f},{y:.1f},{w:.1f},{h:.1f},1,-1,-1,-1\n")
    
            x1, y1, w1, h1 = map(int, bbox)
            cv2.rectangle(frame, (x1, y1), (x1 + w1, y1 + h1), (255, 255, 0), 2)
            cv2.putText(frame, f'ID: {t_id}', (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)

    cv2.imshow('Tracking', frame)
    frame_nr += 1
    if cv2.waitKey(15) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()

result_output_file = f"./{SEQUENCE}.txt"
with open(result_output_file, 'w') as f:
    f.writelines(results_list)